# Initialization

Turning the symlinks into proper paths the system can access. Finding the 8mixmux8 firmware.

In [1]:
import os
from pathlib import Path

base = Path("/home/xilinx/jupyter_notebooks/qick/firmware/projects/qick_tprocv2_111_8mixmux8")
out_dir = base / "out"

# 1. Clean up the non-working links
for name in ["qick_111.bit", "qick_111.hwh"]:
    f = out_dir / name
    if f.exists() or f.is_symlink():
        f.unlink()

# 2. Create actual, native OS-level relative symlinks
os.symlink("../top/top.runs/impl_1/d_1_wrapper.bit", str(out_dir / "qick_111.bit"))
os.symlink("../top/top.gen/sources_1/bd/d_1/hw_handoff/d_1.hwh", str(out_dir / "qick_111.hwh"))

print("Native Linux symlinks successfully recreated!")

Native Linux symlinks successfully recreated!


In [2]:
from qick import QickSoc

# Load the firmware bitstream
soc = QickSoc("/home/xilinx/jupyter_notebooks/qick/firmware/projects/qick_tprocv2_111_8mixmux8/out/qick_111.bit", force_init_clks=True)

soccfg = soc

# Print to check soc details and show code has run
print(soc)

configuring reference clock chips, as requested
LMK04208 clock reference = 122.880 MHz, LMX2594 clock synth = 204.800 MHz
QICK running on ZCU111, software version 0.2.406

Firmware configuration (built Wed Jun 10 15:03:08 2026):

	Global clocks (MHz): tProc dispatcher timing 384.000, RF reference 204.800
	Groups of related clocks: [tProc core clock, tProc timing clock, DAC tile 0, DAC tile 1], [ADC tile 0, ADC tile 2]

	8 signal generator channels:
	0:	axis_sg_mixmux8_v1 - fs=6144.000 Msps, fabric=384.000 MHz
		32-bit DDS, range=1536.000 MHz
		DAC tile 0, blk 0 is DAC228_T0_CH0, or RF board DAC port 0
	1:	axis_sg_mixmux8_v1 - fs=6144.000 Msps, fabric=384.000 MHz
		32-bit DDS, range=1536.000 MHz
		DAC tile 0, blk 1 is DAC228_T0_CH1, or RF board DAC port 1
	2:	axis_sg_mixmux8_v1 - fs=6144.000 Msps, fabric=384.000 MHz
		32-bit DDS, range=1536.000 MHz
		DAC tile 0, blk 2 is DAC228_T0_CH2, or RF board DAC port 2
	3:	axis_sg_mixmux8_v1 - fs=6144.000 Msps, fabric=384.000 MHz
		32-bit DDS, ran

In [4]:
# Jupyter setup boilerplate
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from qick import *
from qick.asm_v2 import AveragerProgramV2 # still necessary even after importing all from QICK

# useful libraries
import numpy as np
import random 
import time
import math

# MixMuxed Signal Generation - Long Period For Spectrum Analysis

Similiar to code from Sho's Github. Implementing / testing the multi-tone generators. Readout is declared for the sake of running but is not used.

In [38]:
# --- Main MixMux8 program ---
class MixMuxProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']
        
        self.declare_gen(ch=gen_ch, nqz=1, ro_ch=ro_ch, 
                         mixer_freq=cfg['mixer_freq'],
                         mux_freqs=cfg['pulse_freqs'], 
                         mux_gains=cfg['pulse_gains'], 
                         mux_phases=cfg['pulse_phases'])
        
        # Declaring unused dynamic readout
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])
        self.add_readoutconfig(ch=ro_ch, name="myro", freq=cfg['pulse_freqs'][0], phase=cfg['ro_phase'], gen_ch=gen_ch)

        self.add_pulse(ch=gen_ch, name="mymixmux", 
                       style="const", 
                       length=cfg["pulse_len"],
                       mask=[0,1,2,3,4,5,6,7], # tones to play must be in the mask
                      )
        
    def _body(self, cfg):        
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'], ddr4=True) # trigger readouts (unused)

        self.pulse(ch=cfg['gen_ch'], name="mymixmux", t=0) # trigger generator

In [39]:
gen_ch = 7 # which gen_ch to use

# --- Config dictionary ---
config = {'gen_ch': gen_ch,
          'ro_ch': 0, # unused readout channel
          'mixer_freq': 0,
          'pulse_freqs': [199.9997, 199.9998, 199.9999, 200.0000, 200.0001, 200.0002, 200.0003, 200.0004], 
          'pulse_gains': [0.125]*8,
          'pulse_phases': [0]*8,
          'ro_phase': 0.0,
          'trig_time': 0.4,
          'pulse_len': 4000000.0,
          'ro_len': 1.9,
         }

# --- Execution ---
prog = MixMuxProgram(soccfg, reps=1, final_delay=0.5, cfg=config)
iq_list = prog.acquire_decimated(soc, rounds=1)

# --- Confirm completion and list frequencies ---
print('Pulses should have successfully ran. Desired tones:')
for i in config['pulse_freqs']:
    print(i)

  0%|          | 0/1 [00:00<?, ?it/s]

Pulses should have successfully ran. Desired tones:
199.9997
199.9998
199.9999
200.0
200.0001
200.0002
200.0003
200.0004


In [233]:
soc.reset_gens()

# MixMuxed Signal Generation - Arbitrary # of Channels

Similiar to the code above, but for an arbitrary number of channels (1-8).

In [5]:
# --- 8MixMux8 program ---
class ManyMixMuxProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_chs = cfg['gen_chs']
                
        # automatically declare gens for each channel - no need to update upon switching channels
        for chl in gen_chs:
            self.declare_gen(ch=chl, nqz=1, ro_ch=ro_ch, 
                             mixer_freq=cfg['mixer_freq'],
                             mux_freqs=cfg['pulse_freqs'][chl], 
                             mux_gains=cfg['pulse_gains'][chl], 
                             mux_phases=cfg['pulse_phases'][chl])
        
        # Declaring unused dynamic readout
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])
        
        # Accesses only the first gen_ch and the first elm of the first pulse_freqs in the dict (arbitrarily, not used)
        self.add_readoutconfig(ch=ro_ch, name="myro", 
                               freq=cfg['pulse_freqs'][0][0],
                               phase=cfg['ro_phase'], gen_ch=gen_chs[0])
        
        for chl in gen_chs:
            self.add_pulse(ch=chl, name=f"mymixmux_{chl}", 
                           style="const", 
                           length=cfg["pulse_lens"][chl],
                           mask=cfg['pulse_masks'][chl], # tones to play must be in the mask
                          )
        
    def _body(self, cfg):        
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'], ddr4=True) # trigger readouts (unused)
        
        for chl in cfg['gen_chs']:
            self.pulse(ch=chl, name=f"mymixmux_{chl}", t=cfg['pulse_t_offs'][chl]) # trigger generator

In [7]:
# --- Generator / pulse data for each generator ---
# Be sure to re-run this code after making changes to it or the crosstalk mx.

gen_chs = [0, 1, 2, 3, 4, 5, 6, 7] # generator ch numbers to play on

# -- Pulse data --

# Pulse frequencies (MHz)
pulse_freqs = [
    [199.3609, 199.5890, 199.8211, 200.7415, 201.1824, 201.5117, 201.7124, 202.4523], # ch 0 freqs
    [199.6970, 200.2906, 200.4310, 200.6500, 200.6843, 201.6794, 201.7234, 202.0635], # ch 1 freqs
    [198.5225, 199.7850, 200.3571, 200.4575, 200.6139, 200.7952, 200.8921, 201.8483], # ch 2 freqs
    [198.8908, 199.1810, 200.0884, 200.3423, 200.3547, 200.7478, 201.0581, 202.0274], # ch 3 freqs
    [198.8066, 198.9941, 199.0758, 199.6125, 199.7600, 199.9050, 200.3570, 202.0002], # ch 4 freqs
    [199.1233, 200.7760, 200.8781, 201.3794, 201.6885, 202.0532, 202.1312, 202.4704], # ch 5 freqs
    [198.8416, 198.9149, 199.3553, 199.7689, 200.5419, 201.0481, 201.5312, 201.5980], # ch 6 freqs
    [198.5342, 199.3402, 199.3985, 200.0257, 201.0533, 201.1749, 201.7475, 202.1034], # ch 7 freqs
]

# Format: index 0 = ch 0 data, index 1 = ch 1 data, ...
pulse_phases = [[0]*8]*8 # phases of tones

# Beware: If you later perform pulse_phases[0][0] = 90, every single channel will suddenly have 90 as its first phase.
# Better to just change everything if needed.

pulse_masks = [[0, 1, 2, 3, 4, 5, 6, 7]]*8 # ch masks

pulse_lens = [4000000]*8 # tone lengths (max is ~4 mil)

pulse_t_offs = [0]*8 # tone time offsets

# Pulse gains (out of 1.0)
orig_gains = [
    [0.5]*8, # ch 0 gains
    [0.5]*8, # ch 1 gains
    [0.5]*8, # ch 2 gains
    [0.5]*8, # ch 3 gains
    [0.5]*88, # ch 4 gains
    [0.5]*8, # ch 5 gains
    [0.5]*8, # ch 6 gains
    [0.5]*8, # ch 7 gains
]

# --- Config dictionary ---
config = {'gen_chs': gen_chs, # generator channels
          
          # Pulse data
          'pulse_freqs': pulse_freqs,
          'pulse_gains': orig_gains,
          'pulse_phases': pulse_phases,
          'pulse_masks': pulse_masks,
          'pulse_lens': pulse_lens,
          'pulse_t_offs': pulse_t_offs,
          
          'mixer_freq': 0, # default 0
          
          # Unused readout channel
          'ro_ch': 0,
          'ro_phase': 0.0,
          'ro_len': 1.9,
          'trig_time': 0.4,
}

In [53]:
# --- Applying dummy crosstalk matrix ---

# Dummy crosstalk matrix (in linear amplitude)
crosstalk_matrix = np.array([
    [1.0000, 0.3300, 0.1089, 0.0359, 0.0119, 0.0039, 0.0013, 0.0004],
    [0.3300, 1.0000, 0.3300, 0.1089, 0.0359, 0.0119, 0.0039, 0.0013],
    [0.1089, 0.3300, 1.0000, 0.3300, 0.1089, 0.0359, 0.0119, 0.0039],
    [0.0359, 0.1089, 0.3300, 1.0000, 0.3300, 0.1089, 0.0359, 0.0119],
    [0.0119, 0.0359, 0.1089, 0.3300, 1.0000, 0.3300, 0.1089, 0.0359],
    [0.0039, 0.0119, 0.0359, 0.1089, 0.3300, 1.0000, 0.3300, 0.1089],
    [0.0013, 0.0039, 0.0119, 0.0359, 0.1089, 0.3300, 1.0000, 0.3300],
    [0.0004, 0.0013, 0.0039, 0.0119, 0.0359, 0.1089, 0.3300, 1.0000],
])

# If in decibels originally, run:
# crosstalk_matrix = 10 ** (crosstalk_dB / 20.0)

inv_crosstalk = np.linalg.inv(crosstalk_matrix)
corrected_gains = inv_crosstalk @ np.array(orig_gains)
corrected_gains = corrected_gains.tolist()

config['pulse_gains'] = corrected_gains
for i in np.round(corrected_gains, 4).tolist():
    print(i)

[0.7519, 0.7519, 0.7519, 0.7519, 0.7519, 0.7519, 0.7519, 0.7519]
[0.5037, 0.5037, 0.5037, 0.5037, 0.5037, 0.5037, 0.5037, 0.5037]
[0.5038, 0.5038, 0.5038, 0.5038, 0.5038, 0.5038, 0.5038, 0.5038]
[0.5038, 0.5038, 0.5038, 0.5038, 0.5038, 0.5038, 0.5038, 0.5038]
[0.5038, 0.5038, 0.5038, 0.5038, 0.5038, 0.5038, 0.5038, 0.5038]
[0.5038, 0.5038, 0.5038, 0.5038, 0.5038, 0.5038, 0.5038, 0.5038]
[0.5037, 0.5037, 0.5037, 0.5037, 0.5037, 0.5037, 0.5037, 0.5037]
[0.7519, 0.7519, 0.7519, 0.7519, 0.7519, 0.7519, 0.7519, 0.7519]


In [80]:
# --- Execution ---
prog = ManyMixMuxProgram(soccfg, reps=1, final_delay=0.5, cfg=config)
iq_list = prog.acquire_decimated(soc, rounds=1)

# Confirm completion and list frequencies
print('Pulses should have successfully begun running.')

  0%|          | 0/1 [00:00<?, ?it/s]

Pulses should have successfully begun running.


In [67]:
# --- Forced Stopping ---
# should_run = False
soc.reset_gens()

# Misc. Useful Stuff

In [ ]:
# Generate random tones
for i in range(8):
    rand_list = [round(random.uniform(198.5, 202.5), 4) for i in range(8)]
    rand_list.sort()
    print(rand_list)
    

In [31]:
# Generate dummy nearest-neighbor decay mx.

n_channels = 8
alpha = 0.33  # 33% leakage to immediate neighbors

# Create a grid of index distances |i - j|
i, j = np.indices((n_channels, n_channels))
distance_matrix = np.abs(i - j)

# Compute decay matrix
dummy = alpha ** distance_matrix
dummy_list = np.round(dummy, 4).tolist()

for i in dummy_list:
    print(i)

[1.0, 0.33, 0.1089, 0.0359, 0.0119, 0.0039, 0.0013, 0.0004]
[0.33, 1.0, 0.33, 0.1089, 0.0359, 0.0119, 0.0039, 0.0013]
[0.1089, 0.33, 1.0, 0.33, 0.1089, 0.0359, 0.0119, 0.0039]
[0.0359, 0.1089, 0.33, 1.0, 0.33, 0.1089, 0.0359, 0.0119]
[0.0119, 0.0359, 0.1089, 0.33, 1.0, 0.33, 0.1089, 0.0359]
[0.0039, 0.0119, 0.0359, 0.1089, 0.33, 1.0, 0.33, 0.1089]
[0.0013, 0.0039, 0.0119, 0.0359, 0.1089, 0.33, 1.0, 0.33]
[0.0004, 0.0013, 0.0039, 0.0119, 0.0359, 0.1089, 0.33, 1.0]
